In [ ]:
# Step 1: clone the Agent Ops Board repository.
# aob_emit / AOB_* are defined by the cell the launcher injects at the top on every start.
import os, shutil, subprocess, sys, time
from pathlib import Path

REPO_URL = "https://github.com/5tannerpl/agent-ops-board.git"
REPO_DIR = Path("/kaggle/working/agent-repo")
API = "http://127.0.0.1:8080"
MODEL_NAME = "Qwen3-4B-Instruct-2507"

gpus = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.strip().splitlines()
aob_emit("notebook_started", {"gpu_count": len(gpus), "gpus": [g.split(" (UUID")[0] for g in gpus]})

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
print("STEP 1 COMPLETE - REPO CLONED", flush=True)

In [ ]:
# Step 2: install the Python dependencies used by workers.check_model and modelcheck.py.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "httpx", "openai", "pydantic", "requests"], check=True)
aob_emit("environment_ready", {"python": sys.version.split()[0]})
print("STEP 2 COMPLETE - DEPENDENCIES INSTALLED", flush=True)

In [ ]:
# Step 3: restore the prebuilt CUDA llama-server from the Dataset (NO REBUILD).
CUDA_DIR = Path("/kaggle/working/llama-cuda")
candidates = [p for p in Path("/kaggle/input").rglob("llama-server")
              if p.is_file() and "agent-ops-llama-cuda-t4" in str(p)]
if not candidates:
    raise FileNotFoundError("llama-server not found: attach the agent-ops-llama-cuda-t4 Dataset")
dataset_root = next(parent for parent in candidates[0].parents if parent.name == "agent-ops-llama-cuda-t4")
if CUDA_DIR.exists():
    shutil.rmtree(CUDA_DIR)
shutil.copytree(dataset_root, CUDA_DIR, symlinks=True)  # keep the versioned shared-library symlinks
LLAMA_SERVER = next(p for p in CUDA_DIR.rglob("llama-server") if p.is_file())
LLAMA_SERVER.chmod(0o755)
LIB_DIRS = sorted({str(p.parent) for p in CUDA_DIR.rglob("*.so*")} | {str(LLAMA_SERVER.parent)})
os.environ["LD_LIBRARY_PATH"] = ":".join(LIB_DIRS + [os.environ.get("LD_LIBRARY_PATH", "")])
print("CUDA CACHE RESTORED - NO REBUILD:", LLAMA_SERVER, flush=True)

In [ ]:
# Step 4: locate the Qwen3-4B GGUF. Do NOT run modelcheck.py here: the server is not started yet.
ggufs = sorted(p for p in Path("/kaggle/input").rglob("*.gguf") if "agent-ops-qwen-model" in str(p))
if not ggufs:
    raise FileNotFoundError("GGUF not found: attach the agent-ops-qwen-model Dataset")
MODEL_PATH = ggufs[0]
print("Model source:", MODEL_PATH, flush=True)
aob_emit("model_located", {"file": MODEL_PATH.name, "size_mb": round(MODEL_PATH.stat().st_size / 1e6)})
print("STEP 4 COMPLETE - MODEL LOCATED", flush=True)

In [ ]:
# Step 5: launch llama-server on GPU 0 and wait for /health HTTP 200.
import urllib.request

server_log = open("/kaggle/working/llama-server.log", "w")
server = subprocess.Popen(
    [str(LLAMA_SERVER), "-m", str(MODEL_PATH), "--host", "127.0.0.1", "--port", "8080",
     "-ngl", "99", "-c", "4096", "--alias", MODEL_NAME],
    stdout=server_log, stderr=subprocess.STDOUT, env={**os.environ, "CUDA_VISIBLE_DEVICES": "0"})

health = None
deadline = time.time() + 600
while time.time() < deadline and health is None:
    if server.poll() is not None:
        raise RuntimeError("llama-server exited:\n" + Path("/kaggle/working/llama-server.log").read_text()[-2000:])
    try:
        with urllib.request.urlopen(API + "/health", timeout=5) as resp:
            health = resp.status if resp.status == 200 else None
    except Exception:
        time.sleep(3)
if health != 200:
    raise RuntimeError("llama-server /health did not return 200 in time")
print(f"HTTP: {health}\nSERVER READY: {API}", flush=True)
aob_emit("model_ready", {"api_health": health, "backend": "llamacpp"})

In [ ]:
# Step 6: real model validation (workers.check_model + modelcheck.py) with signed HTTPS event reporting.
# Send only counts, timings, exit codes and status; no PII, prompts or raw stdout/stderr (printed locally only).
model_env = {**os.environ, "AOB_MODEL_BACKEND": "llamacpp", "AOB_MODEL_URL": API,
             "AOB_MODEL_NAME": MODEL_NAME, "AOB_MODEL_TIMEOUT": "120"}
REQUIRE_EVENT_DELIVERY = True
failed_events = 0


def report(event_type, payload):
    global failed_events
    if not aob_emit(event_type, payload):
        failed_events += 1
        print(f"[EVENT DELIVERY FAILED] {event_type}", flush=True)


def run_check(name, cmd, timeout):
    started = time.perf_counter()
    result = subprocess.run(cmd, cwd=str(REPO_DIR), env=model_env, capture_output=True, text=True, timeout=timeout)
    print(f"=== {name} ===\n{result.stdout}", flush=True)
    if result.stderr:
        print(result.stderr, flush=True)
    return result, round(time.perf_counter() - started, 3)


report("model_validation_started", {"test_runs": 10, "model": MODEL_NAME, "status": "running"})

result, seconds = run_check("workers.check_model (10 runs)",
                            [sys.executable, "-m", "workers.check_model", "--runs", "10"], 300)
report("worker_validation_completed", {
    "test_runs": 10, "duration_seconds": seconds, "exit_code": result.returncode,
    "status": "passed" if result.returncode == 0 else "failed"})
if result.returncode != 0:
    raise RuntimeError("workers.check_model failed")

modelcheck_file = next((rel for rel in ("modelcheck.py", "workers/modelcheck.py") if (REPO_DIR / rel).is_file()), None)
if modelcheck_file is None:
    raise FileNotFoundError("modelcheck.py not found")
result, seconds = run_check("modelcheck.py", [sys.executable, modelcheck_file], 180)
report("modelcheck_completed", {
    "duration_seconds": seconds, "exit_code": result.returncode,
    "status": "passed" if result.returncode == 0 else "failed"})
if result.returncode != 0:
    raise RuntimeError("modelcheck.py failed")

report("model_validation_completed", {"status": "passed", "backend": "llamacpp", "api_health": 200})
print(f"Event delivery failures: {failed_events}", flush=True)
if REQUIRE_EVENT_DELIVERY and failed_events:
    raise RuntimeError("Model tests passed but HTTPS event delivery failed")
print("STEP 6 COMPLETE - MODEL VALIDATION + HTTPS REPORTING PASSED", flush=True)
server.terminate()